# 🧠 Tactile SDF Reconstruction — Google Colab

This notebook trains a **PointNet + SIREN** model using sparse tactile contacts.

**Data Sources:**
- **Grasp Data:** Hugging Face Hub (`jack635/grasp-dataset-curated`)
- **GT Meshes:** Objaverse API (downloads only the 50 required objects)

In [ ]:
# @title 🛠️ Step 1: Setup & Dependencies
!pip install trimesh rtree plotly streamlit scikit-image scikit-learn tqdm huggingface_hub objaverse
!pip install fast-simplification  # For faster mesh decimation

In [ ]:
# @title 📂 Step 2: Download Code & Data
import os
import json
import shutil
import objaverse
from huggingface_hub import snapshot_download

# 1. Clone the tactile-sdf code
if not os.path.exists('tactile-sdf'):
    !git clone https://github.com/635jack/tactile-sdf.git

# 2. Download the grasp dataset
repo_id = "jack635/grasp-dataset-curated"
print(f"📥 Downloading grasp data from {repo_id}...")
snapshot_download(repo_id=repo_id, repo_type="dataset", local_dir="grasp-dataset")

# 3. Download GT meshes from Objaverse
mapping_path = "grasp-dataset/uid_mapping.json"
with open(mapping_path) as f:
    uid_mapping = json.load(f)

print(f"📦 Downloading {len(uid_mapping)} meshes from Objaverse...")
objects = objaverse.load_objects(uids=list(uid_mapping.values()))

# 4. Organize meshes in data/objaverse with correct names
target_dir = "data/objaverse"
os.makedirs(target_dir, exist_ok=True)

for mesh_name, uid in uid_mapping.items():
    if uid in objects:
        src = objects[uid]
        dest = os.path.join(target_dir, f"{mesh_name}.glb")
        shutil.copy(src, dest)
        # print(f"  ✅ Linked {mesh_name}")

print(f"🎉 Setup complete. Meshes ready in {target_dir}")
%cd tactile-sdf

In [ ]:
# @title 📐 Step 3: Run SDF Preprocessing
!python preprocess_sdf.py \
    --n_points 50000 \
    --glb_dir ../data/objaverse \
    --dataset_dir ../grasp-dataset

In [ ]:
# @title 🚀 Step 4: Start Training
!python train.py \
    --epochs 300 \
    --batch_size 8 \
    --dataset_dir ../grasp-dataset

In [ ]:
# @title 📊 Step 5: Visualize Results
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob

runs = sorted(glob.glob('runs/*'))
if runs:
    latest_run = runs[-1]
    img_path = os.path.join(latest_run, 'training_curves.png')
    if os.path.exists(img_path):
        plt.figure(figsize=(15, 10))
        img = mpimg.imread(img_path)
        plt.imshow(img)
        plt.axis('off')
        plt.show()
else:
    print("No training runs found.")